In [24]:
import numpy as np
import pandas as pd
import re

### ============================
### DATA CLEANING & PREPROCESSING
### ============================

In [9]:
df=pd.read_csv("delhi_business_raw.csv")
print(f"\nRaw shape: {df.shape}")
print(f"Columns  : {list(df.columns)}")
df.info()


Raw shape: (11822, 11)
Columns  : ['area', 'category_searched', 'name', 'address', 'rating', 'review_count', 'website', 'phone', 'types', 'business_status', 'price_level']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11822 entries, 0 to 11821
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   area               11822 non-null  object 
 1   category_searched  11822 non-null  object 
 2   name               11822 non-null  object 
 3   address            11822 non-null  object 
 4   rating             11784 non-null  float64
 5   review_count       11784 non-null  float64
 6   website            6019 non-null   object 
 7   phone              10619 non-null  object 
 8   types              11822 non-null  object 
 9   business_status    11822 non-null  object 
 10  price_level        939 non-null    object 
dtypes: float64(2), object(9)
memory usage: 1016.1+ KB


### DROPPING THE DUPLICATE DATA

In [7]:
before = len(df)
df = df.drop_duplicates(subset=["name", "address"], keep="first")
print(f"\nRemoved {before - len(df)} duplicate business listings")


Removed 1781 duplicate business listings


### HANDLE MISSING RATINGS / REVIEW COUNTS

In [ ]:
# A business with no rating/review data can't help us predict rating, and it's a small % of rows (~0.3%), 
# so I dropped those rows rather than invent fake ratings.
before = len(df)
df = df.dropna(subset=["rating", "review_count"])
print(f"Dropped {before - len(df)} rows missing rating/review_count")
df["review_count"] = df["review_count"].astype(int)

Dropped 38 rows missing rating/review_count


### FEATURE ENGINEERING

In [17]:
df["has_website"] = df["website"].notna().astype(int)
df["has_phone"] = df["phone"].notna().astype(int)
df["has_price_info"] = df["price_level"].notna().astype(int)

In [20]:
# mapping price level to an ordinal scale where known, else marking unknown
price_map = {
    "PRICE_LEVEL_INEXPENSIVE": 1,
    "PRICE_LEVEL_MODERATE": 2,
    "PRICE_LEVEL_EXPENSIVE": 3,
    "PRICE_LEVEL_VERY_EXPENSIVE": 4,
}
df["price_level_clean"] = df["price_level"].map(price_map)

In [19]:
df["types"] = df["types"].fillna("")
df["num_types"] = df["types"].apply(lambda x: len([t for t in x.split(",") if t.strip()]))

#### Extracting pincode from the addresses

In [13]:
def extract_pincode(addr):
    match = re.search(r"\b(1\d{5})\b", str(addr))
    return match.group(1) if match else np.nan
 
df["pincode"] = df["address"].apply(extract_pincode)

In [ ]:
# text columns ko clean krna
df["area"] = df["area"].str.strip()
df["category_searched"] = df["category_searched"].str.strip().str.lower()
df["name"] = df["name"].str.strip()

### DATA CLEANING

#### RATING SANITY CHECK

In [14]:
before = len(df)
df = df[(df["rating"] >= 1.0) & (df["rating"] <= 5.0)]
print(f"Dropped {before - len(df)} rows with out-of-range ratings")

Dropped 38 rows with out-of-range ratings


#### OUTLIER CHECK

In [15]:
cap = df["review_count"].quantile(0.99)
outliers = (df["review_count"] > cap).sum()
print(f"\n{outliers} businesses have review_count above the 99th percentile "
      f"({cap:.0f}) -- kept, but flagged with 'review_outlier' column")
df["review_outlier"] = (df["review_count"] > cap).astype(int)


115 businesses have review_count above the 99th percentile (12645) -- kept, but flagged with 'review_outlier' column


#### FINAL COLUMN SELECTION

In [21]:
final_cols = [
    "name", "area", "pincode", "category_searched", "rating", "review_count",
    "review_outlier", "has_website", "has_phone", "has_price_info",
    "price_level_clean", "num_types", "business_status"
]
df_clean = df[final_cols].copy()
 
print(f"\nFinal cleaned shape: {df_clean.shape}")
print(f"\nMissing values remaining per column:")
print(df_clean.isnull().sum())


Final cleaned shape: (11784, 13)

Missing values remaining per column:
name                     0
area                     0
pincode                 48
category_searched        0
rating                   0
review_count             0
review_outlier           0
has_website              0
has_phone                0
has_price_info           0
price_level_clean    10845
num_types                0
business_status          0
dtype: int64


### SAVING THE CLEANED FILE

In [23]:
df_clean.to_csv("delhi_business_clean.csv", index=False)